# Classificação Supervisionada com Gustafson-Kessel (GK) no Dataset DryBean

Este notebook apresenta a implementação de um classificador supervisionado baseado no algoritmo Gustafson-Kessel (GK), aplicado ao dataset DryBean. Todas as etapas seguem o roteiro didático dos experimentos anteriores (KMeans), incluindo importação, pré-processamento, método do cotovelo, implementação do GK supervisionado, avaliação, repetição dos experimentos e análise dos resultados.

## 1. Importação das Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

# Garante que a pasta 'img' existe
os.makedirs('img', exist_ok=True)

## 2. Carregamento e Pré-processamento dos Dados

In [ ]:
# Baixar e carregar o dataset DryBean
import zipfile
import io
import requests
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00602/DryBeanDataset.zip'
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
df = pd.read_excel(z.open('Dry_Bean_Dataset.xlsx'))

# Remover a coluna de saída (classe)
X = df.drop(['Class'], axis=1).values
y = LabelEncoder().fit_transform(df['Class'].values)

# Normalizar os dados
scaler = StandardScaler()
X = scaler.fit_transform(X)

## 3. Divisão dos Dados em Treino e Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

## 4. Definição do Número de Clusters (Método do Cotovelo)

In [ ]:
def gk_objective(X, n_clusters, m=2.0, max_iter=30, random_state=42):
    np.random.seed(random_state)
    n_samples, n_features = X.shape
    U = np.random.dirichlet(np.ones(n_clusters), size=n_samples).T
    q = m
    for _ in range(max_iter):
        V = (U ** q) @ X / np.sum(U ** q, axis=1)[:, None]
        F = np.zeros((n_clusters, n_features, n_features))
        for i in range(n_clusters):
            diff = X - V[i]
            um = (U[i] ** q)[:, None]
            F[i] = (um * diff).T @ diff / np.sum(um)
            F[i] += np.eye(n_features) * 1e-6
        A = np.zeros_like(F)
        for i in range(n_clusters):
            detF = np.linalg.det(F[i])
            if detF <= 0:
                detF = 1e-6
            A[i] = (detF ** (1 / n_features)) * np.linalg.inv(F[i])
        D = np.zeros((n_clusters, n_samples))
        for i in range(n_clusters):
            diff = X - V[i]
            D[i] = np.einsum('ij,jk,ik->i', diff, A[i], diff)
        for i in range(n_clusters):
            denom = np.sum((D[i][:, None] / D.T) ** (1 / (q - 1)), axis=1)
            U[i] = 1.0 / denom
    J = np.sum((U ** m) * D)
    return J

Ks = range(2, 11)
objs = []
for k in Ks:
    objs.append(gk_objective(X_train, n_clusters=k, m=2.0, max_iter=20, random_state=42))
plt.figure(figsize=(8,4))
plt.plot(Ks, objs, 'bx-')
plt.xlabel('Número de clusters')
plt.ylabel('Função Objetivo (GK)')
plt.title('Método do Cotovelo para o GK')
plt.savefig('img/gk_drybean_elbow.png')
plt.show()

## 5. Implementação do Classificador GK Supervisionado

In [ ]:
class GKSupervisionado:
    def __init__(self, n_clusters=7, m=2.0, max_iter=50, tol=1e-5, random_state=0):
        self.n_clusters = n_clusters
        self.m = m
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.cluster_labels_ = None
        self.U_ = None
        self.V_ = None
        self.F_ = None

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        U = np.random.dirichlet(np.ones(self.n_clusters), size=n_samples).T
        q = self.m
        for _ in range(self.max_iter):
            V = (U ** q) @ X / np.sum(U ** q, axis=1)[:, None]
            F = np.zeros((self.n_clusters, n_features, n_features))
            for i in range(self.n_clusters):
                diff = X - V[i]
                um = (U[i] ** q)[:, None]
                F[i] = (um * diff).T @ diff / np.sum(um)
                F[i] += np.eye(n_features) * 1e-6
            A = np.zeros_like(F)
            for i in range(self.n_clusters):
                detF = np.linalg.det(F[i])
                if detF <= 0:
                    detF = 1e-6
                A[i] = (detF ** (1 / n_features)) * np.linalg.inv(F[i])
            D = np.zeros((self.n_clusters, n_samples))
            for i in range(self.n_clusters):
                diff = X - V[i]
                D[i] = np.einsum('ij,jk,ik->i', diff, A[i], diff)
            for i in range(self.n_clusters):
                denom = np.sum((D[i][:, None] / D.T) ** (1 / (q - 1)), axis=1)
                U[i] = 1.0 / denom
        self.U_ = U
        self.V_ = V
        self.F_ = F
        clusters = np.argmax(U, axis=0)
        self.cluster_labels_ = []
        for i in range(self.n_clusters):
            mask = (clusters == i)
            if np.any(mask):
                label = np.bincount(y[mask]).argmax()
            else:
                label = -1
            self.cluster_labels_.append(label)

    def predict(self, X):
        n_samples = X.shape[0]
        n_features = X.shape[1]
        q = self.m
        V = self.V_
        F = self.F_
        A = np.zeros_like(F)
        for i in range(self.n_clusters):
            detF = np.linalg.det(F[i])
            if detF <= 0:
                detF = 1e-6
            A[i] = (detF ** (1 / n_features)) * np.linalg.inv(F[i])
        D = np.zeros((self.n_clusters, n_samples))
        for i in range(self.n_clusters):
            diff = X - V[i]
            D[i] = np.einsum('ij,jk,ik->i', diff, A[i], diff)
        clusters = np.argmin(D, axis=0)
        return np.array([self.cluster_labels_[c] for c in clusters])

    def evaluate(self, X, y_true):
        y_pred = self.predict(X)
        acc = accuracy_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)
        return acc, cm

## 6. Treinamento e Avaliação do Classificador

In [ ]:
# Defina o número de clusters de acordo com o método do cotovelo
n_clusters = 7  # Ajuste conforme o gráfico do cotovelo

clf = GKSupervisionado(n_clusters=n_clusters, m=2.0, max_iter=50, random_state=42)
clf.fit(X_train, y_train)

acc, cm = clf.evaluate(X_test, y_test)
print(f'Acurácia: {acc:.4f}')
print('Matriz de Confusão:')
print(cm)

plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusão - GK (DryBean)')
plt.colorbar()
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.savefig('img/gk_drybean_confusion_matrix.png')
np.save('img/gk_drybean_confusion_matrix.npy', cm)
np.savetxt('img/gk_drybean_confusion_matrix.csv', cm, delimiter=',', fmt='%d')
plt.show()

## 7. Repetição dos Experimentos

In [ ]:
acuracias = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = GKSupervisionado(n_clusters=n_clusters, m=2.0, max_iter=50, random_state=seed)
    clf.fit(X_train, y_train)
    acc, _ = clf.evaluate(X_test, y_test)
    acuracias.append(acc)

acuracias = np.array(acuracias)
print(f'Acurácia média: {acuracias.mean():.4f}')
print(f'Desvio padrão: {acuracias.std():.4f}')

plt.figure(figsize=(8,4))
plt.plot(range(1,31), acuracias, marker='o')
plt.xlabel('Repetição')
plt.ylabel('Acurácia')
plt.title('Acurácia por repetição - GK (DryBean)')
plt.savefig('img/gk_drybean_accuracy_repetitions.png')
np.save('img/gk_drybean_accuracies.npy', acuracias)
np.savetxt('img/gk_drybean_accuracies.csv', acuracias, delimiter=',')
plt.show()

In [ ]:
mse_list = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = GKSupervisionado(n_clusters=n_clusters, m=2.0, max_iter=50, random_state=seed)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_list.append(mse)

mse_array = np.array(mse_list)
print(f'MSE médio: {mse_array.mean():.4f}')
print(f'Desvio padrão do MSE: {mse_array.std():.4f}')

plt.figure(figsize=(8,4))
plt.plot(range(1, 31), mse_array, marker='o')
plt.xlabel('Repetição')
plt.ylabel('Erro Médio Quadrático (MSE)')
plt.title('MSE por repetição - GK (DryBean)')
plt.savefig('img/gk_drybean_mse_repetitions.png')
plt.show()

np.save('img/gk_drybean_mse_repetitions.npy', mse_array)
np.savetxt('img/gk_drybean_mse_repetitions.csv', mse_array, delimiter=',')

## 8. Análise dos Resultados

Os resultados obtidos com o classificador Gustafson-Kessel (GK) no dataset DryBean mostram a capacidade do algoritmo em identificar padrões mesmo em dados de alta dimensionalidade e múltiplas classes.

- **Acurácia média:** Observe o valor apresentado na célula anterior. Valores mais altos indicam boa separação dos grupos, mas lembre-se que o agrupamento é não supervisionado e a correspondência entre clusters e classes pode não ser perfeita.
- **Desvio padrão:** Um desvio padrão baixo indica estabilidade do método em diferentes inicializações. Se o desvio for alto, pode ser interessante aumentar o número de iterações ou testar outras configurações.
- **Matriz de confusão:** Permite visualizar como os clusters encontrados se relacionam com as classes reais. Em agrupamento, é comum que haja trocas de rótulos entre clusters, mas padrões diagonais indicam boa correspondência.
- **MSE:** O erro médio quadrático complementa a análise da acurácia, mostrando o quanto as previsões se afastam dos rótulos reais.

**Dificuldades comuns:**
- O GK pode ser sensível à inicialização e ao número de clusters.
- Em dados com classes muito desbalanceadas ou sobrepostas, a correspondência entre clusters e classes pode ser limitada.
- O tempo de execução é maior que o KMeans tradicional devido ao cálculo das matrizes de covariância adaptativas.

**Sugestão:** Compare os resultados do GK com os obtidos pelo KMeans tradicional e pelo Fuzzy C-Means para avaliar qual abordagem é mais adequada para o seu problema.